In [1]:
import os
import json
import glob
from google.colab import drive

# ==========================================
# 1. 强制重置挂载状态
# ==========================================
print("🔄 正在执行强制挂载流程...")
# 如果之前已经挂载，先卸载以清除残留文件路径冲突
if os.path.exists('/content/drive'):
    drive.flush_and_unmount()

# 重新挂载
drive.mount('/content/drive', force_remount=True)

# ==========================================
# 2. 路径配置与任务预设
# ==========================================
INPUT_DIR = '/content/drive/MyDrive/260509_dataset'
OUTPUT_DIR = '/content/drive/MyDrive/260509_dataset_v2'

# 建筑学采光功能层级先验权重
LIGHTING_PRIORITY = {
    "living_room": 10, "bedroom": 7, "multi_purpose": 6,
    "dining_room": 5, "kitchen": 4, "balcony": 3,
    "bathroom": 2, "utility": 2, "entryway": 1,
    "stairs": 1, "corridor": 0
}

# ==========================================
# 3. 批量升级核心逻辑
# ==========================================
def upgrade_json_dataset(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    json_files = [f for f in glob.glob(os.path.join(input_dir, "*.json")) if not f.endswith('_topology.json')]

    if not json_files:
        print(f"❌ 在 {input_dir} 下未找到 JSON 文件，请确认路径是否正确。")
        return

    print(f"🚀 发现 {len(json_files)} 个文件，开始批量升级...")

    success_count = 0
    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        b_size = data.get("metadata", {}).get("building_size", {})
        max_x, max_y, max_z = b_size.get("x", 0), b_size.get("y", 0), b_size.get("z", 0)

        for room in data.get("rooms", []):
            # 注入层级先验
            room["lighting_priority"] = LIGHTING_PRIORITY.get(room.get("type"), 0)

            # 推导物理采光面 (AABB 边界判定)
            xmin, ymin, zmin = room.get("box_min", [0,0,0])
            xmax, ymax, zmax = room.get("box_max", [0,0,0])
            lighting_surfaces = []

            if ymin == 0: lighting_surfaces.append({"normal": [0.0, -1.0, 0.0], "area": (xmax - xmin) * (zmax - zmin)})
            if ymax == max_y: lighting_surfaces.append({"normal": [0.0, 1.0, 0.0], "area": (xmax - xmin) * (zmax - zmin)})
            if xmin == 0: lighting_surfaces.append({"normal": [-1.0, 0.0, 0.0], "area": (ymax - ymin) * (zmax - zmin)})
            if xmax == max_x: lighting_surfaces.append({"normal": [1.0, 0.0, 0.0], "area": (ymax - ymin) * (zmax - zmin)})
            if zmax == max_z: lighting_surfaces.append({"normal": [0.0, 0.0, 1.0], "area": (xmax - xmin) * (ymax - ymin)})

            room["lighting_surfaces"] = lighting_surfaces

        out_path = os.path.join(output_dir, os.path.basename(file_path))
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        success_count += 1

    print(f"✅ 处理完成！成功转换 {success_count} 个文件，已存放至 {output_dir}")

# 执行任务
upgrade_json_dataset(INPUT_DIR, OUTPUT_DIR)

🔄 正在执行强制挂载流程...
Mounted at /content/drive
🚀 发现 96 个文件，开始批量升级...
✅ 处理完成！成功转换 96 个文件，已存放至 /content/drive/MyDrive/260509_dataset_v2
